# SimPO（Simple Preference Optimization）

> Meng & Xia et al., 2024. DPO 的简化改进：**去掉 reference model**，用长度归一化奖励差。

## 1. 题目背景与动机
DPO 需要同时加载 policy 和 reference 两个模型，显存翻倍；且 reference 的对数似然计算引入额外开销。
SimPO 观察到：在 SFT 起点附近，$\log\frac{\pi_\theta}{\pi_{ref}}$ 可近似为 $\log\pi_\theta$ 的**长度归一化**版本，
于是直接**去掉 reference**，用策略自身对数似然的均值作为隐式奖励：
$$r_{SimPO}(x,y)=\frac{\beta}{|y|}\log\pi_\theta(y|x)\approx\beta\log\frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$$

## 2. 损失
$$\mathcal L_{SimPO}=-\log\sigma\!\left(\frac{\beta}{|y_w|}\log\pi_\theta(y_w|x)-\frac{\beta}{|y_l|}\log\pi_\theta(y_l|x)-\gamma\right)$$
- $|y_w|,|y_l|$：胜者/败者长度，做长度归一化；
- $\gamma>0$：目标奖励差（target reward margin），确保胜者奖励比败者高出至少 $\gamma$，提升区分度；
- $\beta$：温度。

## 3. 与 DPO 对比
| 维度 | DPO | SimPO |
|------|------|------|
| reference model | 需要（冻结） | 不需要 |
| 显存 | 2x 模型 | 1x 模型 |
| 奖励定义 | $\beta\log\frac{\pi_\theta}{\pi_{ref}}$ | $\frac{\beta}{|y|}\log\pi_\theta$ |
| 长度归一化 | 无 | 有 |
| 目标 margin | 无 | 有 $\gamma$ |
| 代表工作 | Rafailov 2023 | Meng 2024 |

## 4. 考察点
- 为什么可以去掉 reference（长度归一化近似）
- $\gamma$ 的作用（避免退化解）
- 与 DPO 的显存/速度对比


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaConfig, LlamaForCausalLM

torch.manual_seed(0)

# ============================================================
# SimPO (Simple Preference Optimization) 核心思想：
# 在 DPO 基础上去掉 reference model，用长度归一化的对数似然作为隐式奖励
# r(x,y) = β/|y| * log π_θ(y|x)
# 损失 = -log σ( r(x,y_w) - r(x,y_l) - γ )
#
# 优势：省一个模型、省显存、训练更快
# 来源: Meng & Xia et al., 2024
# ============================================================

beta = 2.0      # 温度系数（SimPO 推荐 2.0，比 DPO 的 0.1 大）
gamma = 1.0     # 目标奖励差 margin

# 偏好对数据：1 个 prompt + 1 个 chosen + 1 个 rejected
prompt_ids = [1, 2, 3, 4, 5, 6]
chosen_ids = [7, 8, 9, 2]       # 胜者
rejected_ids = [1, 2, 0, 0]     # 败者

# 拼接为两条序列
chosen_full = torch.LongTensor([prompt_ids + chosen_ids])      # [1, 10]
rejected_full = torch.LongTensor([prompt_ids + rejected_ids])  # [1, 10]

# 响应掩码
chosen_mask = torch.zeros_like(chosen_full)
chosen_mask[:, len(prompt_ids):] = 1
rejected_mask = torch.zeros_like(rejected_full)
rejected_mask[:, len(prompt_ids):] = 1

print("== SimPO 数据 ==")
print("chosen 序列:", chosen_full.shape, "  长度:", chosen_mask.sum().item())
print("rejected 序列:", rejected_full.shape, "  长度:", rejected_mask.sum().item())
print("（注意：SimPO 不需要 reference model）")


In [ ]:
# 只需一个策略模型（对比 DPO 需要两个）
policy_model = LlamaForCausalLM(config=LlamaConfig(
    vocab_size=12, num_hidden_layers=1, hidden_size=32
))

print("== 模型清单 ==")
print("SimPO 只需 1 个模型: Policy  ✓")
print("DPO   需要 2 个模型: Policy + Reference")
print("省显存约 50%，且无需 reference 前向计算")


In [ ]:
# ============================================================
# 工具：计算序列的长度归一化对数似然
# r(x, y) = β / |y| * Σ_t log π_θ(y_t | x, y_<t)
# ============================================================
def length_normalized_logprob(model: nn.Module, input_ids: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    """
    返回长度归一化的对数似然（SimPO 隐式奖励）
    input_ids: [1, T]
    response_mask: [1, T]  生成部分为 1
    """
    logits = model(input_ids).logits[:, :-1, :]   # 预测下一个 token
    labels = input_ids[:, 1:]                       # 右移一位
    mask = response_mask[:, 1:]                     # 对齐掩码

    logp = F.log_softmax(logits, dim=-1)
    token_logp = torch.gather(logp, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)  # [1, T-1]

    # 仅在生成部分求和
    seq_logp = (token_logp * mask).sum(dim=1)       # [1]
    seq_len = mask.sum(dim=1).clamp(min=1)          # [1] 生成长度

    # 长度归一化
    normalized_logp = seq_logp / seq_len            # [1]
    return normalized_logp, seq_logp, seq_len

# 计算 chosen / rejected 的长度归一化对数似然
chosen_norm_logp, chosen_total_logp, chosen_len = length_normalized_logprob(
    policy_model, chosen_full, chosen_mask
)
rejected_norm_logp, rejected_total_logp, rejected_len = length_normalized_logprob(
    policy_model, rejected_full, rejected_mask
)

print("== 长度归一化对数似然 ==")
print(f"chosen:  总 logp = {chosen_total_logp.item():.4f}, 长度 = {chosen_len.item()}, 归一化 = {chosen_norm_logp.item():.4f}")
print(f"rejected: 总 logp = {rejected_total_logp.item():.4f}, 长度 = {rejected_len.item()}, 归一化 = {rejected_norm_logp.item():.4f}")


In [ ]:
# ============================================================
# SimPO 损失
#
# L = -log σ( β/|y_w| * log π(y_w) - β/|y_l| * log π(y_l) - γ )
#
#   β/|y| * log π(y) 即隐式奖励 r(x,y)
#   γ 是目标奖励差 margin
# ============================================================
def simpo_loss(chosen_norm_logp, rejected_norm_logp, beta: float = 2.0, gamma: float = 1.0) -> torch.Tensor:
    """
    chosen_norm_logp:  [1] 胜者长度归一化对数似然
    rejected_norm_logp: [1] 败者长度归一化对数似然
    beta:  温度
    gamma: 目标奖励差
    """
    # 隐式奖励差
    reward_diff = beta * (chosen_norm_logp - rejected_norm_logp) - gamma

    # SimPO 损失
    loss = -F.logsigmoid(reward_diff).mean()

    # 隐式奖励
    chosen_reward = beta * chosen_norm_logp
    rejected_reward = beta * rejected_norm_logp

    stats = {
        "loss": loss.item(),
        "chosen_reward": chosen_reward.item(),
        "rejected_reward": rejected_reward.item(),
        "reward_margin": (chosen_reward - rejected_reward).item(),
        "reward_diff_with_gamma": reward_diff.item(),
    }
    return loss, stats

loss, stats = simpo_loss(chosen_norm_logp, rejected_norm_logp, beta=beta, gamma=gamma)

print()
print("== SimPO 损失 ==")
for k, v in stats.items():
    print(f"  {k}: {v:.6f}")


In [ ]:
# ============================================================
# 单步训练 + 对比 DPO
# ============================================================
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-4)

print("== SimPO 单步训练 ==")
optimizer.zero_grad()
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
optimizer.step()
print(f"梯度范数: {grad_norm.item():.6f}")
print(f"损失: {loss.item():.6f}")
print()
print("== SimPO vs DPO 优势 ==")
print("1. 省一个 reference model，显存减半")
print("2. 无需 reference 前向，训练更快")
print("3. 长度归一化缓解长度偏置")
print("4. γ margin 提升胜败区分度，避免退化解")


In [ ]:
# ============================================================
# SimPO vs DPO 对比总结
# ============================================================
summary = """
╔══════════════════════╦═══════════════════════════╦════════════════════════════╗
║        维度          ║            DPO            ║           SimPO            ║
╠══════════════════════╬═══════════════════════════╬════════════════════════════╣
║ reference model      ║ 需要（冻结）              ║ 不需要                     ║
║ 模型数量             ║ 2 (Policy + Reference)    ║ 1 (Policy)                 ║
║ 显存                 ║ 2x                        ║ 1x（省 50%）               ║
║ 隐式奖励             ║ β log(π_θ/π_ref)          ║ β/|y| * log π_θ            ║
║ 长度归一化           ║ 无                        ║ 有                         ║
║ 目标 margin γ        ║ 无                        ║ 有                         ║
║ β 典型值             ║ 0.1                       ║ 2.0                        ║
║ 训练速度             ║ 基准                      ║ 更快（省一次前向）         ║
║ 代表工作             ║ Rafailov 2023             ║ Meng 2024                  ║
╚══════════════════════╩═══════════════════════════╩════════════════════════════╝

SimPO 关键公式:
  隐式奖励: r(x,y) = β/|y| * log π_θ(y|x)
  损失:     L = -log σ( r(x,y_w) - r(x,y_l) - γ )
           = -log σ( β/|y_w| log π(y_w) - β/|y_l| log π(y_l) - γ )
"""
print(summary)


## ✅ 测试验证

In [ ]:
# 验证 SimPO 损失性质
import torch
import torch.nn.functional as F

# SimPO: L = -log σ(β/|yw| log π(yw) - β/|yl| log π(yl) - γ)
# 与 DPO 区别: 无 reference、长度归一化、有 γ margin

beta, gamma = 2.0, 1.0

# 性质1: 当 chosen 和 rejected 的归一化 logp 相等时，loss = -log σ(-γ)
norm_logp = -3.0  # 任意相同值
diff = beta * (norm_logp - norm_logp) - gamma  # = -γ
loss_equal = -F.logsigmoid(torch.tensor(diff)).item()
expected = -F.logsigmoid(torch.tensor(-torch.tensor(gamma))).item()
assert abs(loss_equal - expected) < 1e-6, f"equal case loss mismatch: {loss_equal} vs {expected}"
print(f"  ✓ 相同 logp 时 loss = -log σ(-γ) = {expected:.4f}")

# 性质2: chosen logp 升高 → loss 减小
chosen_logp = -2.0
rejected_logp = -3.0
diff_good = beta * (chosen_logp - rejected_logp) - gamma
loss_good = -F.logsigmoid(torch.tensor(diff_good)).item()
assert loss_good < loss_equal, "better chosen should reduce loss"
print(f"  ✓ chosen 更优时 loss 下降: {loss_good:.4f} < {loss_equal:.4f}")

# 性质3: γ 越大，相同 logp 差时 loss 越大（要求更大 margin）
diff_no_gamma = beta * (chosen_logp - rejected_logp)
loss_no_gamma = -F.logsigmoid(torch.tensor(diff_no_gamma)).item()
assert loss_good > loss_no_gamma, "γ > 0 should increase loss for same diff"
print(f"  ✓ γ margin 增加区分难度: {loss_good:.4f} > {loss_no_gamma:.4f}")

# 性质4: 长度归一化 → 长序列不会因 logp 累积而占优
long_seq_logp = -6.0  # 长序列总 logp 更负
long_seq_len = 10
short_seq_logp = -3.0
short_seq_len = 5
# 归一化后
long_norm = long_seq_logp / long_seq_len  # -0.6
short_norm = short_seq_logp / short_seq_len  # -0.6
assert abs(long_norm - short_norm) < 1e-6, "length normalization should equalize"
print("  ✓ 长度归一化: 相同 per-token logp 的长短序列被公平对待")

print("✅ SimPO 测试通过: γ margin、长度归一化、无 reference 均正确")
